# Scale-up run: n=300, balanced grading labels

Third notebook in the pilot. `pilot.ipynb` ran the original n=100 pipeline;
`02_reasoning_text_entropy.ipynb` tested reasoning-text clustering on GPU.
This one re-runs the **full** pipeline at n=300 with a **rebalanced** sample.

**Why rebalance.** The n=100 sample was 87/13 on `has_error`, which made the
grading task nearly degenerate: a constant "there is an error" predictor scores
0.87, beating the model's 0.75. Balancing removes that confound *and* increases
statistical power, because the model over-predicts "error" -- so clean items are
the ones it gets wrong, pushing the grading-error rate from ~25% toward ~46%.
A 50/50 sample at n=290 gives the same AUROC precision as an unbalanced n=480.

**Why n=300.** At n=300 the 95% CI half-width on a single AUROC is ~0.06,
against ~0.12 at n=100. Four of the five comparisons the pilot ran were
unresolvable at n=100; this is the size at which they become decidable.

**Pre-registered.** `pilot.plotting.SCALEUP_PREREGISTRATION` fixes the
thresholds this run will be judged against, registered 2026-08-02 *before*
running it. The reasoning-arm stratified result it tests (0.756 within the
has_error stratum) was found post-hoc on n=100; this run is what turns it into
a real test. Do not retune those thresholds after seeing these results.

**Run order:** cells 1-4 (install, auth, model, census+sample), then cell 5
(the long one), then 6-7. Cell 8 is optional. The sampling loop is checkpointed
per item to Drive and resumes after a disconnect -- re-run it, don't restart.

In [1]:
# Install cell: GPU-dependent packages only.
# `datasets` is intentionally also in the local requirements.txt -- each
# environment installs its own copy independently, no conflict.
# torch is not installed explicitly: Colab GPU runtimes ship with it preinstalled.
!pip install -q transformers accelerate bitsandbytes datasets qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 70.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 81.1 MB/s eta 0:00:00:00:0100:01


In [3]:
# Auth & code/results access cell.
import json
import os
from getpass import getpass

from huggingface_hub import login

# --- Drive mount first: it holds both the model cache and the token store ---
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

# --- Tokens: entered ONCE, then cached on your Drive ---
# Deliberately not hardcoded in this notebook. This file is tracked in a
# public repo, and GitHub's secret scanning auto-revokes any ghp_ token that
# lands in a public commit -- so an inline token would stop working by
# itself. Drive is private to your account, survives runtime recycling, and
# git never touches it, so you get the same "no retyping" result safely.
TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False  # set True once to replace previously saved tokens


def get_token(name, prompt):
    """Return a saved token, prompting (once) and persisting it if absent."""
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "Stored Hugging Face token does not start with 'hf_'. Set "
        "RESET_TOKENS = True and re-run this cell to replace it."
    )

login(token=HF_TOKEN)
print("Hugging Face login OK")

# --- Clone the repo (code + results live in the same repo for this pilot) ---
# Cloned anonymously: the repo is public, so read access needs no token, and
# keeping the token out of the clone URL means a clone error can never echo
# it into this notebook's saved output. The token is used only to push.
REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"

# Remove any stale clone from a previous (possibly failed) run so this cell
# is safe to re-run -- git clone silently no-ops into a pre-existing
# directory, which would otherwise leave `repo/` incomplete without error.
!rm -rf repo
!git clone -q {REPO_URL} repo

# %pip (not !pip) installs into the *running kernel's* environment -- !pip
# can silently target a different Python install.
%pip install -q -e repo/

# An editable install writes an `__editable__.pilot-*.pth` file into
# site-packages, but .pth files are only processed by the `site` module at
# INTERPRETER STARTUP. The kernel is already running, so it never sees them
# and `import pilot` fails with ModuleNotFoundError even though the install
# reported success. Putting the repo on sys.path directly makes the package
# importable right now, with no kernel restart needed.
import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")

Mounted at /content/drive
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pilot (pyproject.toml) ... done
pilot package imported from: /content/repo/pilot


In [4]:
# Model load cell.
# Start with the 3B model for the first smoke test -- same prompt format and
# code path as the 7B, but noticeably faster to load and run, so early bugs
# get caught cheaply. Swap MODEL_ID to the 7B line below once the pipeline
# runs cleanly end to end on the 3B.
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"  # swap in once the 3B pipeline is clean

# If memory is tight on the 7B, load in 4-bit instead:
# from transformers import BitsAndBytesConfig
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_ID,
#     quantization_config=quantization_config,
#     device_map="auto",
#     cache_dir=DRIVE_MODEL_CACHE,
# )

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    cache_dir=DRIVE_MODEL_CACHE,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

In [ ]:
# Census + balanced sample cell.
#
# Prints the full dataset's has_error composition BEFORE sampling, because the
# achievable balance is the one input the run size depends on and it has never
# been measured -- the 87/13 figure comes from a 100-item sample, not the
# dataset. If the clean pool is too small for a 50/50 sample at n=300,
# load_fermat_balanced preserves the ratio and returns fewer items, warning
# loudly. It does NOT silently return an imbalanced sample.
import logging

import pilot.data

logging.basicConfig(level=logging.WARNING, force=True)

N = 300
SEED = 42
TARGET_ERROR_FRAC = 0.5

# --- Census: what balance can this dataset actually support? ---
import datasets

full = datasets.load_dataset("ai4bharat/FERMAT", split="train")
census = pilot.data.fermat_census(full)
print("FERMAT census:")
for k, v in census.items():
    print(f"  {k:16s} {v}")
print()

max_n = census["max_balanced_n"]
if max_n < N:
    print(
        f"NOTE: a 50/50 sample caps at n={max_n} (clean pool has "
        f"{census['n_clean']} items). Options: run at n={max_n}, or lower "
        f"TARGET_ERROR_FRAC to trade balance for size. Power at n={max_n} "
        f"balanced is still better than n={N} at the natural {census['frac_error']:.0%} rate."
    )
else:
    print(f"A 50/50 sample at n={N} is supported (max balanced n={max_n}).")

del full  # a full copy of the dataset is not needed past the census

sample = pilot.data.load_fermat_balanced(
    n=N, seed=SEED, target_error_frac=TARGET_ERROR_FRAC
)
N = len(sample)  # may be < requested if a pool ran short -- keep the two in sync

n_error = sum(bool(x) for x in sample["has_error"])
print()
print(f"Sample: {N} items, {n_error} with an error, {N - n_error} clean "
      f"({n_error / N:.0%} error rate)")
print(f"Trivial 'always say error' baseline on this sample: "
      f"{max(n_error, N - n_error) / N:.2f}")

In [ ]:
# Sampling loop cell. Same structure as pilot.ipynb's, with two changes:
#   - the sample comes from the balanced census cell above, not load_fermat_sample
#   - transcription and grading can use different K (grading is the arm where
#     more samples plausibly helped: within the has_error stratum, digit-only
#     entropy went 0.756 at K=5 to 0.891 at K=15)
#
# Retry policy is unchanged and deliberate: bounded retries around
# infrastructure failures only, never around parsing. A response that fails to
# parse is real data about that sample, not a transient fault -- retrying until
# it parses would bias every entropy estimate downward.
import gc
import json
import os
import time

import torch
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

import pilot.prompts

K_TRANSCRIPTION = 5
K_GRADING = 5      # set to 15 to test the stratified K result; costs ~10 more
                   # calls/item (~3000 extra at n=300)
TEMP = 0.7
N_TEMP0 = 2
MAX_RETRIES = 3
RETRY_PAUSE_SECONDS = 5
CHECKPOINT_EVERY_GENERATIONS = 200

META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")

INFRA_EXCEPTIONS = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)


def generate(messages, do_sample: bool, temperature: float | None):
    """Run one generation call, retrying only on infrastructure-level failures."""
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            text_prompt = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = processor(
                text=[text_prompt], images=image_inputs, videos=video_inputs,
                padding=True, return_tensors="pt",
            ).to(model.device)

            gen_kwargs = {"max_new_tokens": 512, "do_sample": do_sample}
            if do_sample:
                gen_kwargs["temperature"] = temperature

            with torch.no_grad():
                output_ids = model.generate(**inputs, **gen_kwargs)

            trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
            return processor.batch_decode(
                trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0]
        except INFRA_EXCEPTIONS as exc:
            last_exc = exc
            gc.collect()
            torch.cuda.empty_cache()
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_PAUSE_SECONDS)
    raise last_exc


def atomic_json_dump(obj, path):
    """Write JSON through a temp file so a disconnect cannot corrupt it."""
    tmp_path = f"{path}.tmp"
    with open(tmp_path, "w") as f:
        json.dump(obj, f)
    os.replace(tmp_path, path)


n_items = len(sample)
calls_per_item = (K_TRANSCRIPTION + N_TEMP0) + (K_GRADING + N_TEMP0)

# Checkpoint name carries the balance config too: a run at a different
# TARGET_ERROR_FRAC draws a different sample, and silently resuming from an
# unrelated one would corrupt the results without any error.
CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
checkpoint_prefix = (
    f"scaleup_{MODEL_ID.split('/')[-1]}_n{N}_seed{SEED}"
    f"_bal{int(TARGET_ERROR_FRAC * 100)}_kt{K_TRANSCRIPTION}_kg{K_GRADING}"
)
completed_path = f"{CHECKPOINT_DIR}/{checkpoint_prefix}.jsonl"
partial_path = f"{CHECKPOINT_DIR}/{checkpoint_prefix}.partial.json"
checkpoint_config = {
    "model_id": MODEL_ID, "n": N, "seed": SEED,
    "k_transcription": K_TRANSCRIPTION, "k_grading": K_GRADING,
    "temp": TEMP, "n_temp0": N_TEMP0, "target_error_frac": TARGET_ERROR_FRAC,
}


def entry_is_complete(entry):
    return (
        len(entry.get("transcription_samples_raw", [])) == K_TRANSCRIPTION
        and len(entry.get("grading_samples_raw", [])) == K_GRADING
        and len(entry.get("transcription_temp0_raw", [])) == N_TEMP0
        and len(entry.get("grading_temp0_raw", [])) == N_TEMP0
    )


def entry_matches_sample(entry, sample_item):
    saved_item = entry.get("item", {})
    return all(saved_item.get(k) == sample_item[k] for k in META_FIELDS)


def rewrite_completed_checkpoint(entries):
    with open(completed_path, "w") as f:
        for entry in entries:
            f.write(json.dumps(entry, default=str) + "\n")
        f.flush()


def validate_completed_checkpoint(entries):
    valid_entries = []
    for idx, entry in enumerate(entries[:n_items]):
        if not entry_is_complete(entry):
            print(f"Checkpoint item {idx + 1} is incomplete; resuming from there.")
            break
        if not entry_matches_sample(entry, sample[idx]):
            print(f"Checkpoint item {idx + 1} does not match expected sample order; resuming from there.")
            break
        valid_entries.append(entry)
    if len(valid_entries) != len(entries):
        print(f"Truncating completed checkpoint from {len(entries)} to {len(valid_entries)} valid items.")
        rewrite_completed_checkpoint(valid_entries)
    return valid_entries


raw_results = []
if os.path.exists(completed_path):
    with open(completed_path) as f:
        raw_results = [json.loads(line) for line in f if line.strip()]
    print(f"Found completed-item checkpoint: {len(raw_results)} items")
    raw_results = validate_completed_checkpoint(raw_results)
    print(f"Validated completed items: {len(raw_results)}")

partial_state = None
if os.path.exists(partial_path):
    with open(partial_path) as f:
        candidate = json.load(f)
    if (
        candidate.get("config") == checkpoint_config
        and candidate.get("completed_items") == len(raw_results)
        and candidate.get("item_idx", -1) >= len(raw_results)
    ):
        partial_state = candidate
        print(f"Found partial checkpoint: item {partial_state['item_idx'] + 1}/{n_items}")
    else:
        print("Ignoring stale partial checkpoint that does not match completed items/config.")


def count_entry_calls(entry):
    return sum(
        len(entry.get(field, []))
        for field in ("transcription_samples_raw", "grading_samples_raw",
                      "transcription_temp0_raw", "grading_temp0_raw")
    )


calls_since_partial_checkpoint = 0


def save_partial_checkpoint(item_idx, entry):
    atomic_json_dump(
        {"config": checkpoint_config, "completed_items": len(raw_results),
         "item_idx": item_idx, "entry": entry},
        partial_path,
    )


def maybe_save_partial_checkpoint(item_idx, entry):
    global calls_since_partial_checkpoint
    calls_since_partial_checkpoint += 1
    if calls_since_partial_checkpoint >= CHECKPOINT_EVERY_GENERATIONS:
        save_partial_checkpoint(item_idx, entry)
        calls_since_partial_checkpoint = 0


def run_batch(messages, n, do_sample, temperature, stage, item_idx, outputs, entry, pbar):
    """Draw missing samples for one prompt, resuming from existing outputs."""
    for j in range(len(outputs), n):
        pbar.set_postfix_str(f"item {item_idx + 1}/{n_items} | {stage} {j + 1}/{n}")
        outputs.append(generate(messages, do_sample=do_sample, temperature=temperature))
        maybe_save_partial_checkpoint(item_idx, entry)
        pbar.update(1)
    return outputs


n_done = len(raw_results)
partial_done_calls = count_entry_calls(partial_state["entry"]) if partial_state else 0

if n_done >= n_items:
    print(f"All {n_items} items already done -- nothing to generate.")
else:
    remaining_calls = (n_items - n_done) * calls_per_item - partial_done_calls
    print(f"{n_items - n_done} items left x {calls_per_item} calls "
          f"- {partial_done_calls} resumed = {remaining_calls} generation calls")

    with tqdm(total=remaining_calls, desc="generating", unit="call") as pbar:
        for item_idx, item in enumerate(sample):
            if item_idx < n_done:
                continue

            image = item["image"]
            transcription_messages = pilot.prompts.build_transcription_messages(image)
            grading_messages = pilot.prompts.build_grading_messages(image)

            if partial_state is not None and partial_state["item_idx"] == item_idx:
                entry = partial_state["entry"]
                partial_state = None
            else:
                entry = {
                    "item": {k: item[k] for k in META_FIELDS},
                    "transcription_samples_raw": [],
                    "grading_samples_raw": [],
                    "transcription_temp0_raw": [],
                    "grading_temp0_raw": [],
                }

            run_batch(transcription_messages, K_TRANSCRIPTION, True, TEMP,
                      "transcription", item_idx, entry["transcription_samples_raw"], entry, pbar)
            run_batch(grading_messages, K_GRADING, True, TEMP,
                      "grading", item_idx, entry["grading_samples_raw"], entry, pbar)
            run_batch(transcription_messages, N_TEMP0, False, None,
                      "transcription-t0", item_idx, entry["transcription_temp0_raw"], entry, pbar)
            run_batch(grading_messages, N_TEMP0, False, None,
                      "grading-t0", item_idx, entry["grading_temp0_raw"], entry, pbar)

            raw_results.append(entry)
            with open(completed_path, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n")
                f.flush()
            calls_since_partial_checkpoint = 0
            if os.path.exists(partial_path):
                os.remove(partial_path)

print(f"raw_results: {len(raw_results)} items")

In [ ]:
# Scoring cell.
#
# Perception scoring uses extract_final_answer + canonicalize_math, NOT the
# raw transcription. This is the fix that made the perception arm work at all:
# clustering the full derivation pinned 83/100 items at max entropy because any
# paraphrase counted as disagreement. pilot.ipynb's original scoring cell
# predates that fix and was corrected offline; this notebook applies it inline.
import importlib

import pilot.canonicalize
import pilot.entropy
import pilot.parsing

for m in (pilot.parsing, pilot.canonicalize, pilot.entropy):
    importlib.reload(m)

scored_results = []
for entry in raw_results:
    item = entry["item"]

    transcription_parsed = [
        pilot.parsing.parse_transcription(t) for t in entry["transcription_samples_raw"]
    ]
    grading_parsed = [pilot.parsing.parse_grading(t) for t in entry["grading_samples_raw"]]
    transcription_temp0_parsed = [
        pilot.parsing.parse_transcription(t) for t in entry["transcription_temp0_raw"]
    ]
    grading_temp0_parsed = [pilot.parsing.parse_grading(t) for t in entry["grading_temp0_raw"]]

    # canonical_answer_label = extract_final_answer -> canonicalize_math ->
    # normalize_string. Both the samples and the ground truth MUST go through
    # it: majority_cluster normalizes the labels it returns, so comparing
    # against an un-normalized ground truth silently mismatches every
    # sympy-parsed equation (see the function's docstring -- this scored
    # 36/100 instead of 42/100 on the real data, with no error raised).
    transcription_answers = [
        pilot.canonicalize.canonical_answer_label(t) for t in transcription_parsed
    ]
    temp0_answers = [
        pilot.canonicalize.canonical_answer_label(t) for t in transcription_temp0_parsed
    ]

    perception_entropy = pilot.entropy.cluster_entropy(transcription_answers)
    reasoning_entropy = pilot.entropy.cluster_entropy(
        [None if d is None else str(d) for d in grading_parsed]
    )
    temp0_entropy_transcription = pilot.entropy.cluster_entropy(temp0_answers)
    temp0_entropy_grading = pilot.entropy.cluster_entropy(
        [None if d is None else str(d) for d in grading_temp0_parsed]
    )

    majority_transcription, _ = pilot.entropy.majority_cluster(transcription_answers)
    ground_truth_answer = pilot.canonicalize.canonical_answer_label(item["pert_a"])
    transcription_correct = majority_transcription == ground_truth_answer

    majority_grading, _ = pilot.entropy.majority_cluster(
        [None if d is None else str(d) for d in grading_parsed]
    )
    grading_correct = majority_grading in {"0", "1"} and int(majority_grading) == int(
        item["has_error"]
    )

    scored_results.append({
        "orig_q": item["orig_q"],
        "pert_a": item["pert_a"],
        "has_error": item["has_error"],
        "handwriting_style": item["handwriting_style"],
        "image_quality": item["image_quality"],
        "perception_entropy": perception_entropy,
        "reasoning_entropy": reasoning_entropy,
        "temp0_entropy_transcription": temp0_entropy_transcription,
        "temp0_entropy_grading": temp0_entropy_grading,
        "transcription_correct": transcription_correct,
        "grading_correct": grading_correct,
        "n_transcription_parse_failures": sum(1 for t in transcription_parsed if t is None),
        "n_grading_parse_failures": sum(1 for d in grading_parsed if d is None),
        "all_transcription_samples_raw": entry["transcription_samples_raw"],
        "all_grading_samples_raw": entry["grading_samples_raw"],
        "temp0_transcription_raw": entry["transcription_temp0_raw"],
        "temp0_grading_raw": entry["grading_temp0_raw"],
        "model_id": MODEL_ID,
        "n_items": N,
        "k_transcription": K_TRANSCRIPTION,
        "k_grading": K_GRADING,
        "target_error_frac": TARGET_ERROR_FRAC,
    })

print(f"Scored {len(scored_results)} items.")

# --- Headline numbers against the pre-registered thresholds ---
import pandas as pd

import pilot.plotting
importlib.reload(pilot.plotting)

df = pd.DataFrame(scored_results)
gt = df["has_error"].astype(bool)

summary = {
    "perception": pilot.plotting.bootstrap_auroc_ci(
        df, "perception_entropy", "transcription_correct"),
    "reasoning_pooled": pilot.plotting.bootstrap_auroc_ci(
        df, "reasoning_entropy", "grading_correct"),
    "reasoning_error_stratum": pilot.plotting.bootstrap_auroc_ci(
        df[gt], "reasoning_entropy", "grading_correct"),
    "reasoning_clean_stratum": pilot.plotting.bootstrap_auroc_ci(
        df[~gt], "reasoning_entropy", "grading_correct"),
}
for name, r in summary.items():
    print(f"{name:26s} AUROC {r['auroc']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}] "
          f"n={r['n_items']} n_err={r['n_error']}")

print()
print("Baseline:", pilot.plotting.majority_class_baseline(df, "has_error"))
print()
print("Pre-registered verdicts (thresholds fixed 2026-08-02, before this run):")
for k, v in pilot.plotting.classify_scaleup_result(summary).items():
    print(f"  {k:26s} {v}")

In [ ]:
# Save cell: CSV to Drive first, then repo + push. Distinct filename -- never
# overwrites the n=100 baseline results.
import subprocess
from datetime import datetime, timezone
from getpass import getpass

import pandas as pd

df = pd.DataFrame(scored_results)

model_slug = MODEL_ID.split("/")[-1].lower().replace(".", "")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
csv_name = f"scaleup_n{N}_bal{int(TARGET_ERROR_FRAC * 100)}_{model_slug}_{timestamp}.csv"

# Drive copy BEFORE any git operation: a push can fail for auth or
# fast-forward reasons, and losing hours of GPU output to a git problem
# would be painful. This copy survives regardless of what happens below.
drive_results = "/content/drive/MyDrive/uncertainty-math-vlm/results"
os.makedirs(drive_results, exist_ok=True)
df.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup written to {drive_results}/{csv_name}")

os.makedirs("repo/results", exist_ok=True)
csv_path = f"repo/results/{csv_name}"
df.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(df)} rows)")

_REDACT = []


def git(*args):
    """Run a git command in repo/, surfacing output (redacted) when it fails."""
    result = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{csv_name}")
commit = git("commit", "-m", f"Add scale-up results: {csv_name}")
if commit.returncode != 0:
    print("git commit failed (see above) -- CSV is safe on Drive.")

GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
if not GH_PUSH_TOKEN:
    GH_PUSH_TOKEN = getpass("GitHub token (to push results), then press Enter: ").strip()
_REDACT.append(GH_PUSH_TOKEN)

if not GH_PUSH_TOKEN:
    print("No token given -- skipping push. CSV is saved on Drive and in repo/results/.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
            print("Rebase onto remote failed; attempting push anyway.")
    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed results to the repo.")
    else:
        print("Push failed (see above). The CSV is safe on Drive and in "
              "repo/results/ -- retry the push without re-running the model.")

### Optional: dump the pairwise NLI entailment matrix

Skip this if the session is running long -- the CSV above is already saved and
this can be run later from the raw grading samples it contains.

Worth doing while a GPU is attached: the expensive part of any reasoning-text
clustering variant is the O(K^2) NLI inference. Persisting the raw pairwise
scores once makes every future merge-criterion experiment a free, instant,
offline computation. That matters because the current union-find merge has a
confirmed cascade-merging defect at K>5 (see `pilot/semantic.py`), and testing
replacements for it otherwise needs a fresh GPU session each time.

In [ ]:
# Optional: pairwise NLI entailment matrix dump.
import itertools
import json

import numpy as np
from sentence_transformers import CrossEncoder
from tqdm.auto import tqdm

import pilot.canonicalize
import pilot.parsing

nli = CrossEncoder("cross-encoder/nli-deberta-v3-small", device="cuda")

matrices = []
for entry in tqdm(raw_results, desc="NLI pairs", unit="item"):
    texts = [
        pilot.canonicalize.structural_clean(pilot.parsing.parse_grading_reasoning(s) or "")
        for s in entry["grading_samples_raw"]
    ]
    idx = [i for i, t in enumerate(texts) if t]
    pairs = list(itertools.combinations(idx, 2))
    if not pairs:
        matrices.append({"n_samples": len(texts), "pairs": [], "forward": [], "backward": []})
        continue
    fwd = nli.predict([(texts[i], texts[j]) for i, j in pairs])
    bwd = nli.predict([(texts[j], texts[i]) for i, j in pairs])
    matrices.append({
        "n_samples": len(texts),
        "pairs": [[int(i), int(j)] for i, j in pairs],
        # Full 3-class scores, not just the argmax: a stricter merge criterion
        # may want margins or probabilities, and re-running to get them would
        # defeat the point of saving this at all.
        "forward": np.asarray(fwd).tolist(),
        "backward": np.asarray(bwd).tolist(),
    })

nli_name = f"nli_pairs_n{N}_kg{K_GRADING}_{timestamp}.json"
with open(f"{drive_results}/{nli_name}", "w") as f:
    json.dump({"label_order": "0=contradiction, 1=entailment, 2=neutral",
               "model": "cross-encoder/nli-deberta-v3-small",
               "k_grading": K_GRADING, "items": matrices}, f)
print(f"Wrote {drive_results}/{nli_name}")